In [ ]:
import pandas as pd

train_url = "https://raw.githubusercontent.com/Rswcf/houseprices_kaggle/master/train.csv"
test_url = "https://raw.githubusercontent.com/Rswcf/houseprices_kaggle/master/test.csv"

train = pd.read_csv(train_url)
test = pd.read_csv(test_url)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.head()

In [ ]:

print("Train Info:")
train.info()

print("\nMissing Values:")
print(train.isnull().sum().sort_values(ascending=False).head(20))

print("\nStatistical Summary:")
train.describe()

print("\nSalePrice Summary:")
print(train["SalePrice"].describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(train["SalePrice"], bins=30, kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Sale Price")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=train, x="GrLivArea", y="SalePrice")
plt.title("Living Area vs Sale Price")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=train, x="OverallQual", y="SalePrice")
plt.title("Overall Quality vs Sale Price")
plt.show()

correlations = train.corr(numeric_only=True)["SalePrice"]
print(correlations.sort_values(ascending=False).head(15))

In [ ]:
missing = train.isnull().sum()

missing_percent = (missing / len(train)) * 100

missing_data = pd.DataFrame({
    "Missing Values": missing,
    "Percentage": missing_percent
})

missing_data = missing_data[
    missing_data["Missing Values"] > 0
].sort_values(by="Percentage", ascending=False)

print(missing_data)

In [ ]:

train_clean = train.copy()
test_clean = test.copy()

none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence",
    "MasVnrType", "FireplaceQu", "GarageType",
    "GarageFinish", "GarageQual", "GarageCond",
    "BsmtExposure", "BsmtFinType2", "BsmtQual",
    "BsmtCond", "BsmtFinType1"
]

for col in none_cols:
    train_clean[col] = train_clean[col].fillna("None")
    test_clean[col] = test_clean[col].fillna("None")

zero_cols = [
    "MasVnrArea", "GarageYrBlt"
]

for col in zero_cols:
    train_clean[col] = train_clean[col].fillna(0)
    test_clean[col] = test_clean[col].fillna(0)

for df in [train_clean, test_clean]:
    df["LotFrontage"] = df.groupby("Neighborhood")[
        "LotFrontage"
    ].transform(lambda x: x.fillna(x.median()))

    df["LotFrontage"] = df["LotFrontage"].fillna(
        train_clean["LotFrontage"].median()
    )

train_clean["Electrical"] = train_clean["Electrical"].fillna(
    train_clean["Electrical"].mode()[0]
)

test_clean["Electrical"] = test_clean["Electrical"].fillna(
    train_clean["Electrical"].mode()[0]
)

print("Remaining missing values in train:",
      train_clean.isnull().sum().sum())

print("Remaining missing values in test:",
      test_clean.isnull().sum().sum())

In [ ]:
missing_test = test_clean.isnull().sum()

print(
    missing_test[missing_test > 0]
    .sort_values(ascending=False)
)

In [ ]:
for col in test_clean.columns:
    if test_clean[col].isnull().sum() > 0:

        if test_clean[col].dtype == "object":
            test_clean[col] = test_clean[col].fillna(
                train_clean[col].mode()[0]
            )
        else:
            test_clean[col] = test_clean[col].fillna(
                train_clean[col].median()
            )

print("Remaining missing values:",
      test_clean.isnull().sum().sum())

In [ ]:

for df in [train_clean, test_clean]:

    df["TotalSF"] = (
        df["TotalBsmtSF"] +
        df["1stFlrSF"] +
        df["2ndFlrSF"]
    )

    df["TotalBath"] = (
        df["FullBath"] +
        0.5 * df["HalfBath"] +
        df["BsmtFullBath"] +
        0.5 * df["BsmtHalfBath"]
    )

    df["TotalPorchSF"] = (
        df["OpenPorchSF"] +
        df["EnclosedPorch"] +
        df["3SsnPorch"] +
        df["ScreenPorch"] +
        df["WoodDeckSF"]
    )

    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

    df["YearsSinceRemodel"] = (
        df["YrSold"] - df["YearRemodAdd"]
    )

print(train_clean[
    ["TotalSF", "TotalBath", "TotalPorchSF",
     "HouseAge", "YearsSinceRemodel"]
].head())

In [ ]:
y = train_clean["SalePrice"]

# حذف ستون هدف و شناسه
X = train_clean.drop(["SalePrice", "Id"], axis=1)

X_test = test_clean.drop(["Id"], axis=1)

X = pd.get_dummies(X)
X_test = pd.get_dummies(X_test)

X, X_test = X.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)

print("\nData types:")
print(X.dtypes.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)

print("RMSE:", round(rmse, 2))
print("MAE:", round(mae, 2))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# آموزش مدل
rf_model.fit(X_train, y_train)

# پیش‌بینی
rf_pred = rf_model.predict(X_val)

# ارزیابی
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))
rf_mae = mean_absolute_error(y_val, rf_pred)

print("Random Forest RMSE:", round(rf_rmse, 2))
print("Random Forest MAE:", round(rf_mae, 2))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

rf_log_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_log_model.fit(X_train, y_train_log)

rf_log_pred = rf_log_model.predict(X_val)

rmsle = np.sqrt(
    mean_squared_error(y_val_log, rf_log_pred)
)

print("RMSLE:", round(rmsle, 5))

In [ ]:
# آموزش مدل نهایی با تمام داده‌های آموزشی
from sklearn.ensemble import RandomForestRegressor
import numpy as np

final_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, np.log1p(y))

# پیش‌بینی روی داده‌های تست و تبدیل لگاریتم به قیمت واقعی
test_predictions_log = final_model.predict(X_test)
test_predictions = np.expm1(test_predictions_log)

print("تعداد پیش‌بینی‌ها:", len(test_predictions))
print("کمترین قیمت پیش‌بینی‌شده:", round(test_predictions.min(), 2))


In [ ]:
# ساخت فایل نهایی برای ارسال به Kaggle
submission = pd.DataFrame({
    "Id": test_clean["Id"].values,
    "SalePrice": test_predictions
})

# بررسی صحت فایل قبل از ذخیره
assert len(submission) == len(test_clean), "تعداد ردیف‌ها با داده تست برابر نیست."
assert submission.isnull().sum().sum() == 0, "در فایل خروجی مقدار خالی وجود دارد."
assert (submission["SalePrice"] > 0).all(), "قیمت پیش‌بینی‌شده باید مثبت باشد."

submission.to_csv("submission.csv", index=False)

print("فایل submission.csv با موفقیت ساخته شد.")
print("ابعاد فایل:", submission.shape)
print(submission.head())
